In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [10]:
class CSVImageDataset(Dataset):

    def __init__(self, csv_file):

        df = pd.read_csv(csv_file)

        self.labels = df.iloc[:,0].values
        self.images = df.iloc[:,1:].values.astype("float32")/255.0

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        image = torch.tensor(self.images[idx])
        label = torch.tensor(self.labels[idx],dtype=torch.long)

        return image,label

In [11]:
class TransferNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.feature_extractor = nn.Sequential(

            nn.Linear(784,256),
            nn.ReLU(),

            nn.Linear(256,128),
            nn.ReLU()
        )

        self.classifier = nn.Linear(128,10)

    def forward(self,x):

        x = self.feature_extractor(x)
        x = self.classifier(x)

        return x

In [12]:
def train(model,trainloader,testloader,epochs,lr):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        filter(lambda p:p.requires_grad,model.parameters()),
        lr=lr
    )

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        for images,labels in trainloader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs,labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images,labels in testloader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _,pred = torch.max(outputs,1)

                total += labels.size(0)

                correct += (pred==labels).sum().item()

        print(f"Epoch {epoch+1}/{epochs}  Loss={running_loss/len(trainloader):.4f}  Accuracy={(100*correct/total):.2f}%")

In [13]:
fashion_train = CSVImageDataset("fashion-mnist_train.csv")
fashion_test = CSVImageDataset("fashion-mnist_test.csv")

fashion_trainloader = DataLoader(fashion_train,batch_size=64,shuffle=True)
fashion_testloader = DataLoader(fashion_test,batch_size=64)

model = TransferNet().to(device)

print("\n========== Stage 1 : Fashion-MNIST ==========\n")

train(model,
      fashion_trainloader,
      fashion_testloader,
      epochs=5,
      lr=0.001)


========== Stage 1 : Fashion-MNIST ==========

Epoch 1/5  Loss=0.5307  Accuracy=85.17%
Epoch 2/5  Loss=0.3775  Accuracy=86.85%
Epoch 3/5  Loss=0.3387  Accuracy=87.87%
Epoch 4/5  Loss=0.3113  Accuracy=88.37%
Epoch 5/5  Loss=0.2913  Accuracy=88.47%


In [14]:
for param in model.feature_extractor.parameters():
    param.requires_grad=False

In [15]:
model.classifier = nn.Linear(128,10).to(device)


In [16]:
mnist_train = CSVImageDataset("mnist_train.csv")
mnist_test = CSVImageDataset("mnist_test.csv")

mnist_trainloader = DataLoader(mnist_train,batch_size=64,shuffle=True)
mnist_testloader = DataLoader(mnist_test,batch_size=64)

In [17]:
print("\n========== Stage 2 : Transfer Learning ==========\n")

train(model,
      mnist_trainloader,
      mnist_testloader,
      epochs=3,
      lr=0.001)


========== Stage 2 : Transfer Learning ==========

Epoch 1/3  Loss=1.1831  Accuracy=71.80%
Epoch 2/3  Loss=0.8408  Accuracy=76.31%
Epoch 3/3  Loss=0.7418  Accuracy=78.73%


In [18]:
print("\n===== Stage 3 : Fine Tuning =====")
for param in model.feature_extractor.parameters():
    param.requires_grad=True
train(
    model,
    mnist_trainloader,
    mnist_testloader,
    epochs=5,
    lr=1e-4
)


===== Stage 3 : Fine Tuning =====
Epoch 1/5  Loss=0.3555  Accuracy=93.10%
Epoch 2/5  Loss=0.1906  Accuracy=94.90%
Epoch 3/5  Loss=0.1448  Accuracy=95.70%
Epoch 4/5  Loss=0.1179  Accuracy=96.16%
Epoch 5/5  Loss=0.0992  Accuracy=96.50%
